In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig
from src.config.clipcap_config import (
    CLIPCAP_MAPPER_CLIP_LENGTH,
    CLIPCAP_MAPPER_DROPOUT,
    CLIPCAP_MAPPER_FEEDFORWARD_MULTIPLIER,
    CLIPCAP_MAPPER_NUM_HEADS,
    CLIPCAP_MAPPER_NUM_LAYERS,
    CLIPCAP_MAPPER_PREFIX_LENGTH,
    CLIPCAP_PREFIX_INIT_MEAN,
    CLIPCAP_PREFIX_INIT_STD,
    CLIPCAP_TRAIN_BATCH_SIZE,
    GPT2_MODEL_NAME,
)

EMBEDDING_DIM = int(AutoConfig.from_pretrained(GPT2_MODEL_NAME).hidden_size)

class PrefixTransformerEncoder(nn.Module):
    def __init__(
        self,
        prefix_length=CLIPCAP_MAPPER_PREFIX_LENGTH,
        d_model=EMBEDDING_DIM,
        nhead=CLIPCAP_MAPPER_NUM_HEADS,
        num_layers=CLIPCAP_MAPPER_NUM_LAYERS,
    ):
        super().__init__()
        self.prefix_length = prefix_length
        self.d_model = d_model
        
        # prefix queries, set random initially
        self.prefix_const = nn.Parameter(torch.empty(prefix_length, d_model))
        nn.init.normal_(
            self.prefix_const,
            mean=CLIPCAP_PREFIX_INIT_MEAN,
            std=CLIPCAP_PREFIX_INIT_STD,
        )
        
        # transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=(
                d_model * CLIPCAP_MAPPER_FEEDFORWARD_MULTIPLIER
            ),
            dropout=CLIPCAP_MAPPER_DROPOUT,
            batch_first=True             # set layout to [batch, seq, feature]
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )

    def forward(self, image_tokens): # Tensor[B, 10, 768] -> Tensor[B, 20, 768]
        batch_size = image_tokens.shape[0]

        # Expand prefix queries [10, 768] -> [B, 10, 768]
        prefix_queries = self.prefix_const.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Image Tokens ([B, 10, 768]) + Prefix Queries ([B, 10, 768]) -> [B, 20, 768]
        concat_sequence = torch.cat([image_tokens, prefix_queries], dim=1)
        encoded_sequence = self.transformer_encoder(concat_sequence)
        return encoded_sequence

In [ ]:
# Shape sanity checks
batch_size = CLIPCAP_TRAIN_BATCH_SIZE
test_image_tokens = torch.randn(
    batch_size,
    CLIPCAP_MAPPER_CLIP_LENGTH,
    EMBEDDING_DIM,
)

test_module = PrefixTransformerEncoder(
    prefix_length=CLIPCAP_MAPPER_PREFIX_LENGTH,
    d_model=EMBEDDING_DIM,
    nhead=CLIPCAP_MAPPER_NUM_HEADS,
    num_layers=CLIPCAP_MAPPER_NUM_LAYERS,
)

encoded_out = test_module(test_image_tokens)

print(f"Input Shape (Image Tokens): {test_image_tokens.shape}")
print(f"Output Shape (Encoder Sequence): {encoded_out.shape}")

assert encoded_out.shape == (batch_size, 20, 768), "err: Shape of output not equal [B, 20, 768]"
print("PASS")

Input Shape (Image Tokens): torch.Size([32, 10, 768])
Output Shape (Encoder Sequence): torch.Size([32, 20, 768])
PASS
